# Run the architecture-conformance experiments on a free GPU (Kaggle or Colab)

This notebook runs the **same code** as the project on a free cloud GPU with Ollama, then packages the results for download.

**Current goal: the HARD TIER run (section 6).** Sections 1-5 are one-time setup; run them in order every new session.
The "Later" section at the end holds the confirmatory runs (primary, minimal prompt, held-out wording). Do not run those yet.

**Before you start**
1. **Kaggle:** create a notebook, set *Accelerator* to a GPU, turn *Internet* on. Add the project zip (any file named `architecture-analyzer*.zip`; use the Kaggle zip, which has no `frontend/` folder because Kaggle rejects `__name__` folders) as a dataset or upload it.
   **Colab:** *Runtime > Change runtime type > GPU*, then upload the zip in the Files panel.
2. Sessions end; the **cache** (saved in the results folder) lets you resume by re-running the same cell.


In [ ]:
# ---- settings you may change -------------------------------------------------
import glob, os
ZIP_PATH   = (glob.glob('/kaggle/input/**/architecture-analyzer*.zip', recursive=True)
              + glob.glob('/content/architecture-analyzer*.zip')
              + glob.glob('./architecture-analyzer*.zip'))
ZIP_PATH   = ZIP_PATH[0] if ZIP_PATH else None
# Kaggle automatically UNZIPS uploaded zips, so the project is usually already a folder under /kaggle/input.
SRC_DIR    = None
if not ZIP_PATH:
    hits = glob.glob('/kaggle/input/**/backend/experiments/run_experiments.py', recursive=True)
    if hits:
        SRC_DIR = os.path.dirname(os.path.dirname(os.path.dirname(hits[0])))   # .../<project>/backend/experiments/x.py -> .../<project>
BASE_MODEL = 'qwen2.5:7b-instruct'   # check ollama.com/library for a current 7-8B *instruct* model; name may differ
NUM_CTX    = 4096                    # must exceed the longest prompt (~1,100 tokens) plus the reply
WORK       = '/kaggle/working' if os.path.isdir('/kaggle/working') else '/content'
print('zip:', ZIP_PATH, '| project folder:', SRC_DIR, '| work dir:', WORK)
assert ZIP_PATH or SRC_DIR, 'Project not found: add the Kaggle zip as a dataset (Input panel) and run this cell again. Listing /kaggle/input:\n' + '\n'.join(glob.glob('/kaggle/input/*/*')[:20])


## 1. Check the GPU

In [ ]:
!nvidia-smi || echo 'NO GPU DETECTED - turn on a GPU accelerator, or expect very slow runs'

## 2. Unpack the project and install its Python dependencies

In [ ]:
import subprocess, os, shutil
os.chdir(WORK)
shutil.rmtree('architecture-analyzer', ignore_errors=True)
if ZIP_PATH:
    subprocess.run(['unzip', '-q', '-o', ZIP_PATH, '-d', '.'], check=True)
else:
    shutil.copytree(SRC_DIR, 'architecture-analyzer', copy_function=shutil.copyfile)   # /kaggle/input is read-only; work on a copy
os.chdir(f'{WORK}/architecture-analyzer/backend')
!chmod -R u+w {WORK}/architecture-analyzer
!pip install -q -r requirements-dev.txt
!python -m pytest tests -q --ignore=tests/test_frontend_contract.py 2>&1 | tail -3
print('Expect: 452 passed, 5 skipped (frontend test excluded). If you see 429 or ~378 you are running an OLD zip.')


## 3. Install and start Ollama
If the install line fails, read its error: newer installers need `zstd` (`apt-get install -y zstd`), and some
platforms need `sudo`. The server must answer on port 11434 before continuing.

In [ ]:
!apt-get install -y -q zstd > /dev/null 2>&1 || true
!curl -fsSL https://ollama.com/install.sh | sh
import subprocess, time, urllib.request, os
env = dict(os.environ, OLLAMA_NUM_PARALLEL='1', OLLAMA_CONTEXT_LENGTH=str(NUM_CTX))
server = subprocess.Popen(['ollama', 'serve'], env=env, stdout=open('/tmp/ollama.log', 'w'), stderr=subprocess.STDOUT)
for _ in range(60):
    try:
        urllib.request.urlopen('http://localhost:11434', timeout=2); print('Ollama is up'); break
    except Exception:
        time.sleep(2)
else:
    print(open('/tmp/ollama.log').read()[-2000:]); raise SystemExit('Ollama did not start')

## 4. Pull the model and give it a bigger context window
Ollama silently truncates prompts longer than its context window. The Modelfile below sets `num_ctx`, and temperature 0
keeps answers repeatable.

In [ ]:
!ollama pull {BASE_MODEL}
open('/tmp/Modelfile', 'w').write(f'FROM {BASE_MODEL}\nPARAMETER num_ctx {NUM_CTX}\nPARAMETER temperature 0\n')
!ollama create arch-judge -f /tmp/Modelfile
!ollama show arch-judge | tee {WORK}/model_info.txt

## 5. Smoke test the model through the project's own client (checks JSON output and speed)

In [ ]:
import os, sys, json
os.environ.update(LLM_BASE_URL='http://localhost:11434/v1', LLM_MODEL='arch-judge', LLM_API_KEY='ollama')
sys.path.insert(0, f'{WORK}/architecture-analyzer/backend')
from analysis.semantic.llm_client import client_from_env
from analysis.semantic.llm_analyzer import parse_verdict_json, REVIEW_VERDICTS
c = client_from_env()
r = c.complete('You are a reviewer. Reply with ONLY JSON.',
               'Reply with {"verdict":"violation","confidence":0.9,"explanation":"test","evidence_used":[]}')
print('ok:', r.ok, '| latency %.1fs' % r.latency_s, '| tokens in/out:', r.input_tokens, r.output_tokens)
print('raw:', r.text[:200]); print('parsed:', parse_verdict_json(r.text, REVIEW_VERDICTS))

## 6. HARD TIER (current goal)
Harder-to-read ADRs plus two mixed-decision scenarios; same labels. See `docs/HARD_TIER.md`.

### 6a. Generate it and count the calls (no model calls yet)
Expect `Generated 72 cases`, M3 = 72 calls, M5 = 57.

In [ ]:
%cd {WORK}/architecture-analyzer/backend
!python -m experiments.benchmark.generate --out ../datasets/benchmark/generated_hard --seed 7 --wording hard
!python -m experiments.run_experiments --benchmark ../datasets/benchmark/generated_hard --dry-run --seeds 5 --methods deterministic_adr,full_hybrid,evidence_guided,generic_selective,no_history,no_intent

### 6b. Core run (about 125 new calls, roughly 5-15 minutes)
`--max-calls` stops safely if the session is about to hit a limit; **re-run the same cell to resume from the cache.**

In [ ]:
!python -m experiments.run_experiments --benchmark ../datasets/benchmark/generated_hard --splits dev,test --methods static,deterministic_adr,full_hybrid,evidence_guided,no_history,no_intent,generic_selective --out ../results/hard --max-calls 400 --seeds 5

### 6c. Sanity check: did the model's answers parse?
Good: `parsed ok` almost equal to `calls`, errors 0. If many fail to parse, stop and report it.

In [ ]:
import pandas as pd
p = pd.read_csv('../results/hard/predictions.csv')
p = p[p.llm_called.astype(bool)]
print('calls:', len(p), '| parsed ok:', int(p.parsed_ok.sum()), '| errors:', int(p.error.notna().sum()),
      '| median latency: %.1fs' % p.latency_s.median())
print(open('../results/hard/summary.md').read()[:3500])

### 6d. OPTIONAL: M2, the LLM-only baseline (about 500 calls, 30-45 minutes). Skip if GPU quota is short.

In [ ]:
!python -m experiments.run_experiments --benchmark ../datasets/benchmark/generated_hard --splits dev,test --methods llm_only --out ../results/hard_m2 --max-calls 600

## 7. M7 (extract-then-compute): dev check
M7 keeps M5's escalation but has the model *extract* facts from the ADR (without showing it any date) and lets code compare dates. Iterate on **dev only**. Re-running reuses cached answers, so only the new extraction calls cost anything (about 25-30 calls).

**Look for:** scenario D accuracy (M5 scored 0 of 6), `parsed ok` close to the number of calls, and few `fallback` rows.

In [ ]:
%cd {WORK}/architecture-analyzer/backend
!python -m experiments.run_experiments --benchmark ../datasets/benchmark/generated_hard --splits dev --methods static,evidence_guided,extract_compute --out ../results/hard_m7_dev --max-calls 300 --bootstrap 200

In [ ]:
import pandas as pd
pd.set_option('display.width', 220); pd.set_option('display.max_colwidth', 120)
p = pd.read_csv('../results/hard_m7_dev/predictions.csv')
t = p[p.truth != 'conformant'].copy(); t['correct'] = t.predicted_violation == (t.truth == 'violation')
print(t[t.method.isin(['M5_evidence_guided','M7_extract_compute'])].groupby(['scenario','method']).correct.mean().unstack().round(2))
m7 = p[(p.method == 'M7_extract_compute') & p.llm_called.astype(bool)]
print('M7 calls:', len(m7), '| parsed ok:', int(m7.parsed_ok.sum()), '| fallbacks:', int((m7.outcome_source == 'fallback').sum()))
print(m7[m7.scenario.str.startswith('D')][['case_id','truth','verdict','computed']].to_string(index=False))

## 8. CONFIRMATION on fresh seeds (run ONLY after M7 is frozen and committed)
The confirmation protocol and its pass/fail criteria are in `docs/ANALYSIS_PLAN.md`. Three new benchmark seeds of the hard tier (216 cases), run once, then pooled. About 350 new calls (10-25 minutes). `--max-calls` stops safely; **re-run the same cell to resume**.

### 8a. Generate the seeds

In [ ]:
%cd {WORK}/architecture-analyzer/backend
!python -m experiments.benchmark.generate --out ../datasets/benchmark/generated_hard_s11 --seed 11 --wording hard
!python -m experiments.benchmark.generate --out ../datasets/benchmark/generated_hard_s12 --seed 12 --wording hard
!python -m experiments.benchmark.generate --out ../datasets/benchmark/generated_hard_s13 --seed 13 --wording hard

### 8b. Run the frozen methods once per seed

In [ ]:
M = 'static,deterministic_adr,full_hybrid,evidence_guided,extract_compute'
!python -m experiments.run_experiments --benchmark ../datasets/benchmark/generated_hard_s11 --splits dev,test --methods {M} --out ../results/confirm_s11 --max-calls 400 --bootstrap 100
!python -m experiments.run_experiments --benchmark ../datasets/benchmark/generated_hard_s12 --splits dev,test --methods {M} --out ../results/confirm_s12 --max-calls 400 --bootstrap 100
!python -m experiments.run_experiments --benchmark ../datasets/benchmark/generated_hard_s13 --splits dev,test --methods {M} --out ../results/confirm_s13 --max-calls 400 --bootstrap 100

### 8c. Pool the seeds and read the verdict
The pooling tool refuses to combine runs that used a different model, prompt version or method set.

In [ ]:
!python -m experiments.combine_runs --runs ../results/confirm_s11 ../results/confirm_s12 ../results/confirm_s13 --out ../results/confirm_all
print(open('../results/confirm_all/summary.md').read()[:6000])

## 9. Package for download
Kaggle: *Output* tab. Colab: Files panel. Upload `results_m7.zip` back to the conversation. (The cache file is left out on purpose.)

In [ ]:
!cp {WORK}/model_info.txt ../results/hard/ 2>/dev/null || true
!cd .. && zip -qr {WORK}/results_m7.zip results -x '*.sqlite'
print('Download', f'{WORK}/results_m7.zip')

---
# LATER: confirmatory runs (do NOT run these yet)
These are the primary, minimal-prompt and held-out runs from `docs/ANALYSIS_PLAN.md`. Run them only after the policy and prompts are frozen and the plan is committed to Git.

## LATER 6. Generate the benchmark and count the calls (no model calls yet)

In [ ]:
%cd {WORK}/architecture-analyzer/backend
!python -m experiments.benchmark.generate --out ../datasets/benchmark/generated --seed 7
!python -m experiments.run_experiments --dry-run --seeds 5

## LATER 7. PILOT: one case per scenario (10 cases)
Look at the table, then at `predictions.csv`: is `parsed_ok` true? are `explanation`s sensible? is `latency_s` small?
If most calls fail to parse, fix that **before** spending the full run.

In [ ]:
!python -m experiments.run_experiments --limit-cases 10 --methods evidence_guided,full_hybrid --out ../results/pilot --bootstrap 100
import pandas as pd
p = pd.read_csv('../results/pilot/predictions.csv')
p = p[p.llm_called.astype(bool)]
print('calls:', len(p), '| parsed ok:', int(p.parsed_ok.sum()), '| errors:', int(p.error.notna().sum()),
      '| median latency: %.1fs' % p.latency_s.median())
p[['method','case_id','scenario','truth','verdict','predicted_violation','latency_s','explanation']]

## LATER 8. Dev split (iterate prompts HERE only, never on the test split)
Re-running after a prompt change re-calls only what changed (responses are cached).

In [ ]:
!python -m experiments.run_experiments --splits dev --methods static,full_hybrid,evidence_guided,no_history --out ../results/dev --bootstrap 200

## LATER 9. FULL run, primary setup (freeze prompts and policy first; `docs/ANALYSIS_PLAN.md` says what counts as support)
Without M2 first (99 distinct calls). `--max-calls` stops safely if the session is about to hit a limit; run the same cell again to resume from the cache.

In [ ]:
!python -m experiments.run_experiments --splits dev,test --methods static,full_hybrid,evidence_guided,no_history,no_intent,generic_selective --out ../results/primary --max-calls 400 --seeds 5

### LATER 9b. M2 (LLM on every edge) — 420 calls, the expensive one. Run it ONCE, on the primary setup.

In [ ]:
!python -m experiments.run_experiments --splits dev,test --methods llm_only --out ../results/primary_m2 --max-calls 600

## LATER 9c. Robustness runs (99 calls each; skip M2)
* **Minimal prompt**: same data, all scenario-specific guidance removed from the system prompts.
* **Held-out wording**: the same repositories with the ADR decision language rewritten. **Commit `docs/ANALYSIS_PLAN.md` to Git before this cell** so the plan provably came first.

In [ ]:
!python -m experiments.run_experiments --splits dev,test --prompt-variant minimal --methods static,full_hybrid,evidence_guided,no_history,no_intent,generic_selective --out ../results/minimal --max-calls 400 --seeds 5

In [ ]:
!python -m experiments.benchmark.generate --out ../datasets/benchmark/generated_heldout --seed 7 --wording heldout
!python -m experiments.run_experiments --benchmark ../datasets/benchmark/generated_heldout --splits dev,test --methods static,deterministic_adr,full_hybrid,evidence_guided,no_history,no_intent,generic_selective --out ../results/heldout --max-calls 400 --seeds 5

## LATER 10. Package everything for download

In [ ]:
!cp {WORK}/model_info.txt ../results/primary/ 2>/dev/null || true
!cd .. && zip -qr {WORK}/results.zip results
print('Download', f'{WORK}/results.zip', '(Kaggle: Output tab; Colab: Files panel).')
print(open('../results/primary/summary.md').read()[:3000])